In [20]:
import json

f=open("dataset_articles.json", encoding="utf8" )
data =json.load(f)

filtered_data=[]
for doc in data:
    if doc["abstract"] and len(doc["abstract"])>50 and doc["keywords"] and len(doc["keywords"])>1:
        filtered_data.append(doc)

print(len(filtered_data))

523


In [22]:
import re

def normalize_keywords(s):
    s = s.lower()
    keywords = re.split(r"[;,]",s)
    return {k.strip() for k in keywords}

abstract_pairs=[]
for doc1 in filtered_data:
    for doc2 in filtered_data:
        if doc1 == doc2:
            break
        else:
            keywords1 = normalize_keywords(doc1["keywords"])
            keywords2 = normalize_keywords(doc2["keywords"])
            score = len(set(keywords1) & set(keywords2))
            abstract_pairs.append((doc1["abstract"], doc2["abstract"],score))

print(len(abstract_pairs))

136503


In [20]:
from collections import Counter

Counter([a[2] for a in abstract_pairs])

Counter({0: 135759, 1: 706, 2: 34, 3: 3, 4: 1})

Balancear

In [8]:
from sklearn.utils import resample

majority_class = [pair for pair in abstract_pairs if pair[2]==0]
minority_class = [pair for pair in abstract_pairs if pair[2]!=0]

undersampled_majority_class = resample(majority_class,
                                       replace=False,     # Don't duplicate samples
                                       n_samples= len(minority_class),  # Match minority 
                                       random_state=42)

balanced_pairs = undersampled_majority_class + minority_class
print(Counter([a[2] for a in balanced_pairs]))

Counter({0: 744, 1: 706, 2: 34, 3: 3, 4: 1})


score normalization

In [10]:
def normalize_score(score):
    if score == 0:
        return 0
    if score == 1:
        return 0.5
    if score == 2:
        return 0.75
    if score >= 3:
        return 0.85
    
norm_balanced_pairs = [(a1, a2, normalize_score(s)) for a1, a2, s in balanced_pairs]

In [15]:
#Criar train, test split estratificados
from sklearn.model_selection import train_test_split
scores = [score for _, _, score in norm_balanced_pairs]
train_data, test_data = train_test_split(
    norm_balanced_pairs,
    test_size=0.2,
    random_state=42,
    stratify=scores
)

In [23]:
Counter([score for a1,a2,score in train_data])

Counter({0: 595, 0.5: 565, 0.75: 27, 0.85: 3})

In [24]:
Counter([score for a1,a2,score in test_data])

Counter({0: 149, 0.5: 141, 0.75: 7, 0.85: 1})

In [5]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.losses import CoSENTLoss

# Load a model to train/finetune
model = SentenceTransformer("neuralmind/bert-base-portuguese-cased", model_kwargs={"torch_dtype": "float32"})

# Initialize the CoSENTLoss
# This loss requires pairs of text and a float similarity score as a label
loss = CoSENTLoss(model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9329.74it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from sentence_transformers import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="models/meu_modelo",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    run_name="mpnet-base-all-nli-triplet",  # Will be used in W&B if `wandb` is installed
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list([{"abstract1": a1, "abstract2": a2, "score": score} for a1, a2, score in train_data])
test_dataset = Dataset.from_list([{"abstract1": a1, "abstract2": a2, "score": score} for a1, a2, score in test_data])

In [ ]:
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sentence_transformers.sentence_transformer.evaluation import TripletEvaluator
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction

# 1. Load a model to finetune with 2. (Optional) model card data
model = SentenceTransformer(
    "neuralmind/bert-base-portuguese-cased"
)

# 4. Define a loss function
loss = CoSENTLoss(model)

# 5. (Optional) Specify training arguments
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="models/my_models",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    run_name="mpnet-base-all-nli-triplet",  # Will be used in W&B if `wandb` is installed
)

# 6. (Optional) Create an evaluator & evaluate the base model
dev_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=eval_dataset["abstract1"],
    sentences2=eval_dataset["abstract2"],
    scores=eval_dataset["score"],
    main_similarity=SimilarityFunction.COSINE,
)
dev_evaluator(model)

# 7. Create a trainer & train
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)
trainer.train()

# 8. Save the trained model
model.save_pretrained("models/mpnet-base-all-nli-triplet/final")

Inference

In [9]:
model = SentenceTransformer("lfcc/medlink-bi-encoder")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2186.14it/s]


In [15]:
query= """S/
Identidicação: género feminino, 24 anos.

AP:
# sem antecedentes conhecidos.

MH:
# contraceptivo oral

HDA: Recorre ao serviço de urgência por nódulos violáceos dolorosos na região pré-tibial bilateral, que estenderam a toda a perna e coxa, com 1 mês de evolução. Três semanas 
antes do início do quadro, realizou a 2ª dose da vacina contra SARS-CoV-2 Comirnaty® (Pfizer-BioNTech), a mesma que realizou um mês antes. Sem outras queixas, nomeadamente 
sugestivas de patologia auto-imune ou síndrome onstitucional.
O/
Exame físico- apirética, sem alterações da orofaringe ou adenopatias, observando-se lesões nodulares em ambas as coxas e pernas, de cor violácea, eritematosas, com 2 cm de maior 
eixo, dolorosas à palpação.
A/
# Analiticamente- leucocitose de 12,6 x 109/L com neutrofilia, proteína C reactiva 210 mg/L e velocidade de sedimentação 42 mm/h.
# Anticorpo anti-estreptolisina O ne-gativo, serologias para sífilis e vírus Ebstein-Barr, hepatite B e C e VIH negativas. Painel imunológico, com pesquisa de anticorpos anti-nucleares (ANA), 
anticorpos anti-double--stranded DNA (dsDNA), anticorpos itoplasmáticos anti--neutrófilos (ANCA), complemento, factor reumatóide e anticorpos anti-péptido citrulinado cíclico, negativos.
# Marcadores de lesão hepática e função renal sem alterações.
# Hemoculturas negativas.
# TC TAP sem alterações.

Assim, assumido eritema nodoso secundário à administração da vacina contra SARS-CoV-2.
Iniciada prednisolona 60 mg/dia.

Evolução:
# Uma semana depois houve melhoria clínica significativa e diminuição acentuada dos parâmetros inflamatórios, pelo que se iniciou redução gradual de corticoterapia, com resolução 
completa do quadro ao fim de duas semanas.
# Passados seis meses, a doente teve doença ligeira por SARS-CoV-2 e uma semana depois surgiram lesões com características idênticas às do episódio anterior. Iniciou novo ciclo de 
corticoterapia, com resolução do quadro. Pela exuberância das lesões, a doente foi aconselhada a não repetir imunização com vacina de mRNA contra SARS-CoV-2.
# Até à data, não voltou ter infecção por SARS-CoV-2 nem teve ressurgimento das lesões cutâneas.
"""

In [23]:
abstracs =[data["abstract"] for data in filtered_data]
keywords = [data["keywords"] for data in filtered_data]

In [13]:
embeddings = model.encode(abstracs)

In [16]:
query_embeddings = model.encode(query)
query_embeddings

array([-1.23656034e-01, -6.48911655e-01, -1.72993407e-01, -5.81772447e-01,
        1.36336279e+00,  4.50945735e-01, -6.02725446e-02,  2.47151196e-01,
        2.37767205e-01, -2.37282127e-01, -3.93353164e-01,  1.86005682e-01,
       -9.12615180e-01, -1.82173222e-01, -5.82212329e-01,  2.67573670e-02,
        4.67485487e-01,  2.81753652e-02, -3.51277351e-01, -4.83166754e-01,
       -1.42211899e-01, -7.37901181e-02, -1.10576522e+00, -3.54278296e-01,
        4.95530993e-01,  6.78348780e-01,  2.10324258e-01,  3.98231775e-01,
        3.23868334e-01, -2.44629532e-02, -5.30105710e-01,  7.77703762e-01,
        2.78456748e-01,  3.83300006e-01, -1.94360405e-01,  3.65977854e-01,
        6.23878278e-02, -2.25418553e-01,  1.46749854e-01, -5.42914987e-01,
       -7.16272056e-01, -5.25603779e-02, -1.89457431e-01, -5.11648953e-01,
       -3.60374868e-01,  1.66355133e-01, -4.50664200e-02,  4.07361925e-01,
       -3.27847540e-01,  4.46251258e-02, -1.92230254e-01,  1.82759315e-01,
        3.48214984e-01,  

In [24]:
from sentence_transformers import util
import torch

cosine_scores = util.pytorch_cos_sim(query_embeddings, embeddings)

ir_res = torch.topk(cosine_scores, k=15)

for cos, idx in zip(ir_res.values[0], ir_res.indices[0]):
    print("score:", cos)
    print("Abstract:", abstracs[idx])
    print("Keywords: ", keywords[idx])
    print("-"*20)

score: tensor(0.6316)
Abstract: Resumo A síndrome de Evans é rara, existindo poucos casos descritos de associação a vacina mRNA SARS-CoV-2, nenhum deles em Portugal. Doente do sexo masculino de dezoito anos, admitido pordor abdominal, icterícia, urina escura e cansaço. Na avaliação apresentava anemia hemolítica autoimune por anticorpos quentes, tendo desenvolvido posteriormente trombocitopenia autoimune. Dos antecedentes destacava-se vasculite IgA aos vinte meses e administração de vacina mRNA SARS-CoV-2 nove dias antes da admissão. Foram excluídas infeções virais, doenças autoimunes/linfoproliferativas, imunodeficiências, fármacos e trombofilias. Apesar de não ter apresentado resposta inicial a corticoterapia e imunoglobulina, verificou-se evolução favorável após a introdução de rituximab e ciclofosfamida. Dada a relação temporal e a exclusão de outras causas, foiefetuado o diagnóstico de síndrome de Evans provavelmente desencadeado por vacina mRNA SARS-CoV-2. É o primeiro caso descri